# Myna v1-scaled Training

**Architecture:** d_model=1024, n_layers=16 (~310–420M params) targeting **macro ≥ 0.80**.
**Config:** `--config scaled`, anti-prior ON, batch=16, max-steps=8000.
**Cost estimate:** $100–$150 on A100/RTX A6000 (20–30 h runtime).
**Target:** G1b stretch gate pass.

In [ ]:
# Install dependencies
!pip install uv torch==2.x.x --quiet
print("Dependencies installed")

In [ ]:
# Unpack repo archive
import tarfile, shutil, os
if os.path.exists('myna_repo.tar.gz'):
    with tarfile.open('myna_repo.tar.gz', 'r:gz') as tar:
        tar.extractall()
    print("Repo extracted")
else:
    print("No myna_repo.tar.gz found; using mounted volume if available")
    # Kaggle mounts user datasets; ensure path is accessible
    !ls /kaggle/input/

In [ ]:
# Run training
!uv run python -m myna.train \
  --suite data/decision-v2-pilot \
  --train-splits calibration development \
  --anti-prior on \
  --batch 16 \
  --max-steps 8000 \
  --seed 0 \
  --config scaled \
  --out runs/v1_scaled_420m \
  --steps 8000 \
  --lr 6e-4 \
  --save-every 50 \
  --eval-every 250 \
  --device auto

In [ ]:
# Verify artifacts created
import json
model_path = "runs/v1_scaled_420m/model_last.pt"
if os.path.exists(model_path):
    blob = __import__('torch').load(model_path, map_location='cpu')
    cfg = blob['cfg']
    print(f"Model config: {cfg}")
    print(f"Saved step: {blob.get('step', 'N/A')}")
else:
    print("Model not saved; check training logs above")